# AMC 2026 – Round 2 · Notebook: a tactical deviation with discretionary equity investing
Financial Modeling (USI) · TA session 4

## Goal of today
Starting from the Round 1 SAA, add a few individual stocks chosen with the Class 5 tools. Size them with Treynor-Black (Class 3 Part II) and check them against the tracking-error budget.

## Round 2 rules (from the guidelines)
- Sell part of the SAA and invest the proceeds in **individual stocks** that you select. The **SAA does not change**: it is the reference.
- Choose the weights with the **Treynor-Black model, with the SAA as the passive portfolio**.
- Identify **catalysts** that could drive the stock price toward its fundamental value.
- **Ex-ante tracking error against the SAA between 2% and 5% per year:** last 60 months, annualized.
- Report the ex-ante **volatility** and the **1-month 5% VaR** of the new portfolio next to those of the SAA.
- **Checkpoint 1 (alphas):** 3–6 stocks with alpha and justification; weights ∝ α/σ²(ε), zero for negative α.
- **Checkpoint 2 (valuation dossier):** method and why, intrinsic value with inputs, margin of safety, value-trap check, catalyst.
- Positions go into Bloomberg (PRTU) on **7–8 November** with the prices of Friday 6 November. Report due **Sunday 8 November, 23:59**.

The data come from `01_download_data.ipynb` (CSV files in this folder).

## 0. Settings

In [1]:
import numpy as np                     # arrays and matrix algebra
import pandas as pd                    # tables with dates
pd.set_option("display.float_format", "{:.4f}".format)

# SAA of Round 1: the ERC portfolio of notebook 03_SAA (each group uses its own SAA here)
W_SAA = pd.Series({"SPY": 0.086, "EFA": 0.074, "EEM": 0.070, "IEF": 0.177,
                   "LQD": 0.124, "TIP": 0.200, "GLD": 0.097, "DBC": 0.159})   # the rest (1.3%) is cash
ETFS   = list(W_SAA.index)
STOCKS = ["NKE", "HD", "GIS"]          # Nike, Home Depot, General Mills 

CAPITAL = 100_000_000                  # USD 100 million at the start of the challenge
NAV     = CAPITAL                      # ⚠️ in the challenge: the value of YOUR portfolio at the close of Fri 6 Nov (PORT), not 100M
WINDOW  = 60                           # months, as in Round 1
ERP     = 0.06114                      # Equity risk premium (Bloomberg CRP, as in Class 5) ⚠️ Compute again!
TE_MIN, TE_MAX, TE_TARGET = 0.02, 0.05, 0.04   # tracking-error budget of Round 2 (target: safely inside)

## 1. Data, monthly returns and the SAA (Round 1 recap) §1–§2

In [2]:
prices = pd.read_csv("r2_adj_prices.csv",   index_col=0, parse_dates=True)[ETFS + STOCKS]   # adjusted
close  = pd.read_csv("r2_close_prices.csv", index_col=0, parse_dates=True)[ETFS + STOCKS]   # raw closing
divs   = pd.read_csv("r2_dividends.csv",    index_col=0, parse_dates=True)
rates  = pd.read_csv("r2_rates.csv",        index_col=0, parse_dates=True) / 100            # % -> decimal

# Same steps as Round 1: month-end prices -> simple monthly returns
R  = prices.ffill().resample("ME").last().pct_change().dropna()
rf = (rates["DGS1MO"] / 12).resample("ME").last().reindex(R.index).ffill()   # monthly T-bill rate
Re = R.sub(rf, axis=0)                 # excess returns R - rf
est = Re.iloc[-WINDOW:]                # the last 60 months

print("months available:", len(R), "| estimation window:", est.index[0].date(), "->", est.index[-1].date())

# The SAA: benchmark and passive portfolio (Round 1 recap)
R_S = est[ETFS] @ W_SAA                # monthly excess return of the SAA (cash earns 0 -> about -rf, negligible)

mu_S    = R_S.mean() * 12              # expected excess return of the SAA, E(R_S^e)
sigma_S = R_S.std() * np.sqrt(12)      # ex-ante volatility of the SAA
var_S   = 1.645 * sigma_S / np.sqrt(12)   # 1-month 5% VaR
sr_S    = mu_S / sigma_S               # Sharpe Ratio of the SAA

print(f"SAA: E(R^e) = {mu_S:.2%}  vol = {sigma_S:.2%}  VaR = {var_S:.2%} (USD {var_S*CAPITAL:,.0f})  Sharpe = {sr_S:.2f}")

months available: 129 | estimation window: 2021-11-30 -> 2026-10-31
SAA: E(R^e) = 2.11%  vol = 7.94%  VaR = 3.77% (USD 3,768,716)  Sharpe = 0.27


## 3. Valuation inputs: CAPM cost of equity §3

$$k = r_f + \beta\,\text{ERP}, \qquad \beta = \frac{\text{Cov}(R_i, R_m)}{\text{Var}(R_m)}$$

**k = cost of equity:** the return shareholders require given the risk. Risk-free (10-year Treasury, long horizon like the dividends) + ERP (Equity Risk Premium) (6.114%, Bloomberg CRP, same for all stocks) scaled by beta (Market Sensitivity). It is the **discount rate** of the DDM (§4) and the **benchmark** of the alpha (§5). The 1-month T-bill is used only for monthly excess returns and the Sharpe Ratio.

| Input | What it is |
|---|---|
| P0 | Last raw close (6 Nov in the challenge) |
| D0 | Dividends per share of the last 12 months (add buybacks per share if the company returns cash mainly through buybacks!); then D1 = D0(1+g) |
| g1 | Dividend growth in the first phase: analysts' consensus growth forecast (look it up in Bloomberg, e.g. `EE`) |
| g2, T | Long-run growth and length of the first phase: **your assumptions**, justify them ! (g2 < k)! |

**Choose the valuation model and justify your choice:**
- **Gordon** (one g forever): for a mature, stable company, e.g. **GIS**.
- **Two-stage** (g1 for T years, then g2 forever): when today's growth is not the long-run one. 
- **Relative valuation** (EPS × average P/E of the competitors): as a second opinion.

In [3]:
# Inputs of each stock: P0 = last raw close, D0 = dividends of the last 12 months, beta vs SPY (60 months)
inp = pd.DataFrame(index=STOCKS)
inp["P0"] = close[STOCKS].ffill().iloc[-1]
inp["D0"] = divs[STOCKS].loc[divs.index > close.index[-1] - pd.Timedelta(days=365)].sum()
inp["beta"] = [est[s].cov(est["SPY"]) / est["SPY"].var() for s in STOCKS]

# Cost of equity, CAPM for each stock: k = rf + beta * ERP
rf10 = rates["DGS10"].dropna().iloc[-1]
inp["k"] = rf10 + inp["beta"] * ERP

# ASSUMPTIONS (placeholders): justify them in the report
inp["g1"] = [0.10, 0.06, 0.02]     # first-phase growth
inp["g2"] = [0.04, 0.04, 0.02]     # long-run growth (< k)
inp["T"]  = [5, 5, 5]              # years of the first phase (not used for GIS)
inp["method"] = ["two-stage", "two-stage", "Gordon"]   # NKE, HD -> two-stage; GIS (mature) -> Gordon

print(f"10-year Treasury yield: {rf10:.2%}")
inp

10-year Treasury yield: 5.00%


,P0,D0,beta,k,g1,g2,T,method
NKE,37.5450,1.6400,1.0369,0.1134,0.1000,0.0400,5,two-stage
HD,320.3950,9.2900,0.9557,0.1084,0.0600,0.0400,5,two-stage
GIS,33.2150,2.4400,0.0102,0.0506,0.0200,0.0200,5,Gordon


## 4. Intrinsic value: Gordon and two-stage DDM §4

**Gordon** (one growth rate forever, needs k > g): for a company whose growth today is already its long-run growth.
*GIS (General Mills):* a mature packaged-food company (cereals, snacks) with low, stable growth and a dividend paid without interruption for over a century. One constant $g$ (Gordon) describes it well.

$$V_0 = \frac{D_0(1+g)}{k-g}$$

**Two-stage** (g1 for T years, then g2 forever): when today's growth is temporarily different from the long-run one.
- *NKE (Nike) – through a turnaround period:* a strong company in a bad phase. Sales and margins fell due to pressure from tariffs; it is trying to recover under a new CEO.
- *HD (Home Depot) – cyclical downturn:* Sells home improvement and renovation products. High mortgage rates reduce home sales and renovations, keeping earnings below normal.

$$V_0 = \sum_{t=1}^{T}\frac{D_0(1+g_1)^t}{(1+k)^t} + \frac{1}{(1+k)^T}\cdot\frac{D_T(1+g_2)}{k-g_2}$$

First term: discounted dividends of the recovery years. Second term: Gordon value at year T, discounted to today.

**Margin of safety:**

$$\text{MoS} = \frac{V_0}{P_0} - 1$$

Positive = the stock looks cheap (buy it); negative = overvalued (short it -> weight 0).

In [4]:
def two_stage(D0, k, g1, g2, T):
    """V0 = sum of the discounted dividends of years 1..T + discounted Gordon terminal value at T."""
    t  = np.arange(1, T + 1)
    D  = D0 * (1 + g1) ** t                         # dividends of years 1..T
    TV = D[-1] * (1 + g2) / (k - g2)                # Gordon value at year T
    return (D / (1 + k) ** t).sum() + TV / (1 + k) ** T

assert (inp["k"] > inp["g2"]).all(), "Gordon needs k > g"

inp["V_gordon"]   = inp["D0"] * (1 + inp["g2"]) / (inp["k"] - inp["g2"])
inp["V_2stage"]   = [two_stage(r.D0, r.k, r.g1, r.g2, int(r.T)) for r in inp.itertuples()]
inp["V0"]         = np.where(inp["method"] == "Gordon", inp["V_gordon"], inp["V_2stage"])
inp["upside"]     = inp["V0"] / inp["P0"] - 1   # margin of safety vs the current price

inp[["P0", "V_gordon", "V_2stage", "method", "V0", "upside"]]

,P0,V_gordon,V_2stage,method,V0,upside
NKE,37.5450,23.2381,29.7820,two-stage,29.7820,-0.2068
HD,320.3950,141.1813,153.6208,two-stage,153.6208,-0.5205
GIS,33.2150,81.2631,81.2631,Gordon,81.2631,1.4466


### 4b. Relative valuation: comparables §4
Compare the stock with its competitors: price it at the **average forward P/E of the comparables** (Class 5).

$$V_i^{comp} = EPS_i^{forward} \times \overline{P/E}_{comparables}$$

A second opinion next to the DDM: if both say the stock is cheap, the case is stronger

In [5]:
# Relative valuation (Class 5): price each stock at the average forward P/E of its competitors
mult  = pd.read_csv("r2_multiples.csv", index_col=0)   # forward EPS and forward P/E, stocks + competitors (from 01)
NAMES = {"NKE": "Nike", "LULU": "Lululemon", "DECK": "Deckers (Hoka, UGG)", "ADDYY": "Adidas",
         "HD": "Home Depot", "LOW": "Lowe's", "FND": "Floor & Decor", "TSCO": "Tractor Supply",
         "GIS": "General Mills", "KHC": "Kraft Heinz", "CAG": "Conagra Brands", "CPB": "Campbell's"}
mult.insert(0, "company", mult.index.map(NAMES))      # company name next to the ticker
peers = mult[mult["role"] == "peer"]
display(peers[["company", "group", "forwardPE"]])      # check: the comparables of each stock and their forward P/E

comp = pd.DataFrame(index=STOCKS)
comp["EPS_fwd"]  = mult.loc[STOCKS, "forwardEps"]                    # analysts' consensus EPS for next year
comp["PE_peers"] = peers.groupby("group")["forwardPE"].mean()        # average forward P/E of the competitors
comp["V_comp"]   = comp["EPS_fwd"] * comp["PE_peers"]                # value at the peers' multiple
comp["P0"]       = inp["P0"]                                         # current price
comp["upside_relative_valuation"] = comp["V_comp"] / comp["P0"] - 1
comp["upside_DDM"]  = inp["upside"]                                  # from the DDM (§4)

# Verdict: both methods agree -> over/undervalued; otherwise mixed
both_pos = (comp["upside_relative_valuation"] > 0) & (comp["upside_DDM"] > 0)
both_neg = (comp["upside_relative_valuation"] < 0) & (comp["upside_DDM"] < 0)
comp["verdict"] = np.where(both_pos, "undervalued", np.where(both_neg, "overvalued", "mixed: check"))
comp

,company,group,forwardPE
ticker,,,
LULU,Lululemon,NKE,12.2851
DECK,"Deckers (Hoka, UGG)",NKE,9.7718
ADDYY,Adidas,NKE,14.2152
LOW,Lowe's,HD,15.3620
FND,Floor & Decor,HD,22.6000
TSCO,Tractor Supply,HD,15.5405
KHC,Kraft Heinz,GIS,11.5091
CAG,Conagra Brands,GIS,9.3637
CPB,Campbell's,GIS,11.1741


,EPS_fwd,PE_peers,V_comp,P0,upside_relative_valuation,upside_DDM,verdict
NKE,1.5550,12.0907,18.8016,37.5450,-0.4992,-0.2068,overvalued
HD,16.0248,17.8342,285.7890,320.3950,-0.1080,-0.5205,overvalued
GIS,3.1523,10.6823,33.6734,33.2150,0.0138,1.4466,undervalued


**Reading the results:**
- **NKE, HD:** V0 < P0 with both the DDM and the comparables → **overvalued** → don't buy (weight 0) (cannot short sell).
- **GIS:** V0 > P0 with both methods → **undervalued** → buy candidate. But the comparables give only +2.5% (≈ fair value), while the DDM gives +143%: check the inputs and the value trap before buying.

## 5. From valuation to alpha: does the gap close? §5
Class 5 gives two cases, both over a **1-year horizon**.

**Your judgment** (justify it in the dossier), for each stock:
- **Catalyst** → the gap closes within 1 year → use $\alpha^{\text{closed}}$
- **No catalyst** → the gap never closes → use $\alpha^{\text{never}}$

$$\alpha^{\text{closed}} = (1+k)\cdot \frac{V_0 - P_0}{P_0}, \qquad \alpha^{\text{never}} = \frac{D_1}{P_0} + g - k$$

**My hypotheses:**
- **NKE, HD:** the gap closes ($\alpha^{\text{closed}}$). Their alpha is negative anyway, so the weight is 0.
- **GIS:** the gap does **not** close ($\alpha^{\text{never}}$): (Checkpoint 2) sales are falling, debt is rising and earnings are expected to fall, so it is cheap for a reason.

In [6]:
# Compute alpha if the gap closes and alpha if it never closes, for all stocks (annual, 1-year horizon)
inp["gap"]          = (inp["V0"] - inp["P0"]) / inp["P0"]
inp["alpha_closed"] = (1 + inp["k"]) * inp["gap"]
inp["alpha_never"]  = inp["D0"] * (1 + inp["g2"]) / inp["P0"] + inp["g2"] - inp["k"]

# Our judgment: NKE, HD: the gap closes (anyway negative alpha -> weight 0)
#                GIS: the value-trap check (Checkpoint 2) says it is cheap for a reason -> the gap does NOT close
inp["gap closes?"] = [True, True, False]

inp["alpha"] = np.where(inp["gap closes?"], inp["alpha_closed"], inp["alpha_never"])    # alpha used in Treynor-Black
inp[["V0", "P0", "gap", "k", "alpha_closed", "alpha_never", "gap closes?", "alpha"]]

,V0,P0,gap,k,alpha_closed,alpha_never,gap closes?,alpha
NKE,29.7820,37.5450,-0.2068,0.1134,-0.2302,-0.0280,True,-0.2302
HD,153.6208,320.3950,-0.5205,0.1084,-0.5770,-0.0383,True,-0.5770
GIS,81.2631,33.2150,1.4466,0.0506,1.5198,0.0443,False,0.0443


## Checkpoint 2: value-trap check
A cheap stock is either **mispriced** (the gap will close) or a **value trap**: it is cheap because its earnings are about to fall (Class 5).

Compute value trap only for stocks that look cheap --> **positive alpha** (here only GIS). 

Three steps:

**1. Compute 6 metrics** (the 4 quality characteristics of Class 5):

$$\frac{V}{B} = \frac{\text{ROE} \times \text{payout ratio}}{k - g}$$

($V$ = intrinsic value of equity, $B$ = book value of equity.) A high $V/B$ is justified only by **high ROE, high growth $g$, low risk $k$ and profits paid to shareholders**. 


**A cheap stock that fails on these is a value trap**.

| Characteristic | Metric | Good if |
|---|---|---|
| Profitability | ROE; FCF margin (FCF / revenue) | high |
| Growth | revenue growth; gross margin | high |
| Safety | debt / equity | low |
| Payout | (dividends + buybacks) / FCF | low (> 1 = paid with debt) |

**2. Check each metric on two dimensions** (last fiscal year):
- **Level:** vs the median of the competitors.
- **Trend:** vs the stock's own average of the previous years. 

Worse than peers **and** worsening = **red flag**.

**3. Check expected earnings:** forward vs trailing EPS. If analysts expect earnings to fall (forward EPS < trailing EPS) --> value trap

In [7]:
# VALUE-TRAP CHECK (Checkpoint 2): 6 metrics (the 4 quality characteristics of Class 5), each checked on two dimensions
#   LEVEL: our stock's last fiscal year vs the median of its competitors (the same peers as §4b)
#   TREND: our stock's last fiscal year vs its own average of the previous years
# + the direct test of a value trap: do analysts expect earnings to fall? (trailing vs forward EPS)
STOCK = "GIS"                                                     # the stock we check (try also "NKE" or "HD")

fund  = pd.read_csv("r2_fundamentals.csv", index_col=[0, 1], parse_dates=["Date"])
mult  = pd.read_csv("r2_multiples.csv", index_col=0)
PEERS = mult[(mult["group"] == STOCK) & (mult["role"] == "peer")].index

def metrics(t):
    """The 6 metrics of firm t, one row per fiscal year."""
    f   = fund.loc[t].dropna(subset=["Total Revenue"])              # drop empty years
    eq  = f["Stockholders Equity"].where(f["Stockholders Equity"] > 0)   # negative equity: ROE and D/E are meaningless
    rev = f["Total Revenue"]
    payout = -(f["Cash Dividends Paid"].fillna(0) + f["Repurchase Of Capital Stock"].fillna(0))   # dividends + buybacks
    return pd.DataFrame({
        "ROE":            f["Net Income"] / eq,                  # 1. profitability
        "FCF margin":     f["Free Cash Flow"] / rev,             # 1. real cash per dollar of sales (defined even with a loss)
        "revenue growth": rev / rev.shift(1) - 1,                # 2. growth
        "gross margin":   f["Gross Profit"] / rev,               # 2. growth not bought with price cuts
        "debt / equity":  f["Total Debt"] / eq,                  # 3. safety
        "payout / FCF":   payout / f["Free Cash Flow"],          # 4. payout paid with cash (> 1: with debt)
    })

HIGHER_IS_BETTER = pd.Series({"ROE": True, "FCF margin": True, "revenue growth": True,
                              "gross margin": True, "debt / equity": False, "payout / FCF": False})

def check(x, ref, good, bad):
    """good if x beats ref in the right direction, bad otherwise, n/a if a number is missing."""
    return np.where(x.isna() | ref.isna(), "n/a",
                    np.where((x > ref) == HIGHER_IS_BETTER, good, bad))

m  = metrics(STOCK)
vt = pd.DataFrame({STOCK:              m.iloc[-1],                                             # last fiscal year
                   "peer median":      pd.concat([metrics(p).iloc[-1] for p in PEERS], axis=1).median(axis=1),
                   "own past average": m.iloc[:-1].mean()})
vt["LEVEL vs peers"] = check(vt[STOCK], vt["peer median"],      "✅ better",    "⚠️ worse")
vt["TREND"]          = check(vt[STOCK], vt["own past average"], "✅ improving", "⚠️ worsening")
print(f"{STOCK}, fiscal year {m.index[-1].year}, peers: {', '.join(PEERS)}")
display(vt.round(2))

# The value-trap question: will a low P/E correct through P going up or through E going down?
eps_t, eps_f = mult.loc[STOCK, "trailingEps"], mult.loc[STOCK, "forwardEps"]
print(f"EPS: trailing {eps_t:.2f} -> forward {eps_f:.2f}: analysts expect earnings to {'rise ✅' if eps_f > eps_t else 'fall ⚠️'}")

red = vt.index[(vt["LEVEL vs peers"] == "⚠️ worse") & (vt["TREND"] == "⚠️ worsening")]
print("Red flags (worse than peers AND worsening):", ", ".join(red) if len(red) else "none")

GIS, fiscal year 2026, peers: KHC, CAG, CPB


,GIS,peer median,own past average,LEVEL vs peers,TREND
ROE,-0.0100,-0.1400,0.2500,✅ better,⚠️ worsening
FCF margin,0.0900,0.0900,0.1200,✅ better,⚠️ worsening
revenue growth,-0.0500,-0.0300,-0.0200,⚠️ worse,⚠️ worsening
gross margin,0.3400,0.2800,0.3400,✅ better,⚠️ worsening
debt / equity,1.8900,1.1400,1.4100,⚠️ worse,⚠️ worsening
payout / FCF,1.1200,0.7000,1.2400,⚠️ worse,✅ improving


EPS: trailing -1.64 -> forward 3.15: analysts expect earnings to rise ✅
Red flags (worse than peers AND worsening): revenue growth, debt / equity


**Reading the results (GIS):**
- **Red flags:** revenue falling and leverage (debt / equity) rising, both worse than peers.
- **EPS:** in the last 12 months GIS booked a one-time loss (trailing EPS −1.64), so going to 3.15 looks like growth. But in FY2024–FY2025 EPS was 4.1–4.3 (`r2_fundamentals.csv`): from about 4 to 3.15, earnings are falling.
- **Verdict:** cheap for a reason → **value-trap risk**: use $\alpha^{\text{never}}$ (§5) or drop GIS.

## 6. Compute the index model with the SAA as the index §6

For each stock, regress its monthly excess returns (last 60 months) on those of the SAA:

$$R^e_{i,t} = a_i + \beta_i\, R^e_{S,t} + \varepsilon_{i,t}$$

- $\beta_i$: how much the stock moves with the SAA.
- $\sigma^2(\varepsilon_i) = 12 \cdot \text{Var}(\varepsilon_{i,t})$: the **idiosyncratic risk**, the part the SAA does not explain.
- $a_i$ is a historical average (the intercept): it is **not** our alpha $\alpha_i$ (that comes from future information).


In [8]:
# R_i^e = a_i + beta_i * R_S^e + eps_i   on the last 60 months (Class 3 Part II, eq. 8, with the SAA as the index)
reg = pd.DataFrame(index=STOCKS)
for s in STOCKS:
    b, a = np.polyfit(R_S, est[s], 1)               # slope (beta vs SAA) and intercept
    eps  = est[s] - (a + b * R_S)                   # residuals
    reg.loc[s, "beta_S"] = b
    reg.loc[s, "a_hist"] = a * 12                   # historical intercept: NOT the alpha!
    reg.loc[s, "sigma2_eps"] = eps.var() * 12       # annualized residual variance

reg

,beta_S,a_hist,sigma2_eps
NKE,1.9897,-0.3083,0.0734
HD,1.5049,-0.0450,0.0468
GIS,0.4700,-0.1088,0.0443


## 7. Treynor-Black: weights of the stocks and size of the active portfolio §7

**1. Weights inside the active portfolio A:** more alpha per unit of idiosyncratic risk means more weight; negative alpha means weight 0 (long-only).

$$w_i = \frac{\alpha_i/\sigma^2(\varepsilon_i)}{\sum_j \alpha_j/\sigma^2(\varepsilon_j)}$$

**2. Characteristics of A:**

$$\alpha_A = \sum_i w_i\alpha_i, \qquad \beta_A = \sum_i w_i\beta_i, \qquad \sigma^2(\varepsilon_A) = \sum_i w_i^2\,\sigma^2(\varepsilon_i)$$

**3. Size of A vs the SAA** (Class 3 Part II, eqs. 11–12):

$$w_A^0 = \frac{\alpha_A/\sigma^2(\varepsilon_A)}{E(R^e_S)/\sigma_S^2}, \qquad w_A^* = \frac{w_A^0}{1 + (1-\beta_A)\,w_A^0}$$

The rest stays in the SAA (the complement):

$$w_S^* = 1 - w_A^*$$

In [9]:
alpha = inp["alpha"]                   # expected alpha from §5 (valuation x catalyst), NOT the historical a_i

# Weights inside the active portfolio: proportional to alpha / sigma^2(eps), zero if alpha < 0 (long-only)
score = (alpha / reg["sigma2_eps"]).clip(lower=0)
assert (score > 0).any(), "No stock has a positive alpha: choose other stocks."
w = score / score.sum()

n_pos = int((w > 0).sum())
if n_pos < 3:
    print(f"⚠️ Only {n_pos} stock with a positive alpha: Checkpoint 1 asks for 3 to 6 stocks. What I have done is not enough!\n")

alpha_A  = (w * alpha).sum()
beta_A   = (w * reg["beta_S"]).sum()
s2_eps_A = (w**2 * reg["sigma2_eps"]).sum()

assert mu_S > 0, "Treynor-Black needs a positive expected excess return of the SAA"
# Size of the active portfolio (Class 3 Part II, eqs. 11-12)
w_A0     = (alpha_A / s2_eps_A) / (mu_S / sigma_S**2)
w_A_star = w_A0 / (1 + (1 - beta_A) * w_A0)

print(f"alpha_A = {alpha_A:.2%}  beta_A = {beta_A:.2f}  sigma(eps_A) = {np.sqrt(s2_eps_A):.2%}")
print(f"Treynor-Black w_A* = {w_A_star:.2f}")
if not 0 < w_A_star <= 1:
    print("-> w_A* is above 100% or negative: with alphas of this size the formula asks for a huge bet. The TE budget will decide (next cell).")
w.rename("weight within A").to_frame()

⚠️ Only 1 stock with a positive alpha: Checkpoint 1 asks for 3 to 6 stocks. What I have done is not enough!

alpha_A = 4.43%  beta_A = 0.47  sigma(eps_A) = 21.05%
Treynor-Black w_A* = 0.26


,weight within A
NKE,0.0000
HD,0.0000
GIS,1.0000


**Reading the results:**
- **Inside A:** only GIS has a positive alpha, so A = 100% GIS.
- **Size of A:** with $\alpha_A = 4.35\%$ (GIS, gap does not close) Treynor-Black gives $w_A^* = 0.26$: 26% in GIS, 74% in the SAA.
- **But** 26% would give a TE above our 4% target: the tracking-error budget (§8) caps the size of A.

## 8. Tracking-error budget, volatility and VaR vs the SAA §8

The tracking error is the volatility of the *difference* between our portfolio's return and the SAA's return, annualized:

$$TE = \text{sd}(R_{new} - R_S)\sqrt{12}$$

TE = 0 means we hold exactly the SAA. It is the risk of our *bet*, not the total risk. Round 2 asks for 2% ≤ TE ≤ 5%.

The new portfolio is $(1 − w_A)·SAA + w_A·A$, so the difference from the SAA is

$$R_{new} - R_S = w_A\,(R_A - R_S) \quad\Longrightarrow\quad TE = w_A \cdot \underbrace{\text{sd}(R_A - R_S)\sqrt{12}}_{\text{TE per unit of } A}$$

The TE grows in proportion to $w_A$. So the budget turns into a lower and an upper bound on the size:

$$w_A^{lo} = \frac{2\%}{\text{sd}(R_A - R_S)\sqrt{12}}, \qquad w_A^{hi} = \frac{4\%}{\text{sd}(R_A - R_S)\sqrt{12}}$$

We use 4% as an upper bound instead of 5% to stay safely inside the budget.

In [10]:
R_A = est[STOCKS] @ w                            # monthly excess return of the active portfolio

# New portfolio = (1 - w_A) * SAA + w_A * A  ->  R_new - R_S = w_A * (R_A - R_S)
# TE = sd(R_new - R_S) * sqrt(12) is proportional to w_A: the budget gives a lower and an upper bound on w_A
te_per_unit = (R_A - R_S).std() * np.sqrt(12)
w_A_lo = TE_MIN / te_per_unit                    # smallest w_A allowed (TE = 2%)
w_A_hi = TE_TARGET / te_per_unit                 # largest w_A we accept (TE = 4%, safely below 5%)

w_A = float(np.clip(w_A_star, w_A_lo, w_A_hi))   # Treynor-Black, pushed inside the budget if needed
R_new = (1 - w_A) * R_S + w_A * R_A

te      = (R_new - R_S).std() * np.sqrt(12)
sigma_N = R_new.std() * np.sqrt(12)
risk = pd.DataFrame({"SAA": [sigma_S, 1.645 * sigma_S / np.sqrt(12), 0.0],
                     "New": [sigma_N, 1.645 * sigma_N / np.sqrt(12), te]},
                    index=["Ex-ante vol", "1-month 5% VaR", "Tracking error"])
print(f"w_A* = {w_A_star:.2f} | budget: {w_A_lo:.2%} (TE 2%) to {w_A_hi:.2%} (TE {TE_TARGET:.0%})  ->  chosen w_A = {w_A:.2%}")

risk

w_A* = 0.26 | budget: 9.32% (TE 2%) to 18.64% (TE 4%)  ->  chosen w_A = 18.64%


,SAA,New
Ex-ante vol,0.0794,0.0816
1-month 5% VaR,0.0377,0.0387
Tracking error,0.0000,0.0400


## 9. Checkpoint 1: the alphas, the weights and the information ratio §9

**What Checkpoint 1 asks (one page):**
1. **3–6 stocks** with their alpha and its justification (valuation + catalyst).
2. **Weights** ∝ $\alpha_i/\sigma^2(\varepsilon_i)$, weight 0 if the alpha is negative.
3. The **information ratio**, what it adds to the Sharpe Ratio of the SAA, and how credible it is.

Class 3 Part II, eqs. 15–16:

$$IR_i = \frac{\alpha_i}{\sigma(\varepsilon_i)}, \qquad IR_A^2 = \sum_{i \in A} IR_i^2, \qquad SR_P^2 = SR_S^2 + IR_A^2$$

(The sum runs only over the stocks you hold, $\alpha_i > 0$: the negative $IR_i$ of NKE and HD do not enter.)

**Our case, step by step:**
1. **Alpha (§5):** positive only for GIS, $\alpha = 4.35\%$ (gap does not close) → A = 100% GIS.
2. **Treynor-Black (§7):** $w_A^* = 26\%$.
3. **TE budget (§8):** TE 2%–4% allows $w_A$ between 9.3% and 18.5% → $w_A = 18.5\%$.
4. **Portfolio:** 81.5% SAA + 18.5% GIS; TE 4.0%, vol 7.9% → 8.2%.
5. **IR and Sharpe:** $IR_A = 0.21$ (credible: below Buffett's 0.64); Sharpe 0.26 → 0.33.

In [11]:
# Checkpoint 1: alphas, Treynor-Black weights and information ratio
cp1 = pd.DataFrame({"alpha": alpha,
                    "sigma2(eps)": reg["sigma2_eps"],
                    "weight in A": w,                                   # ∝ alpha / sigma2(eps), 0 if alpha < 0 (from §7)
                    "IR_i": alpha / np.sqrt(reg["sigma2_eps"])})         # alpha_i / sigma(eps_i)
display(cp1)

# Information ratio of the active portfolio (eq. 15) and what it adds to the SAA Sharpe Ratio (eq. 16)
IR_A = alpha_A / np.sqrt(s2_eps_A)
SR_P = np.sqrt(sr_S**2 + IR_A**2)
print(f"IR_A = {IR_A:.2f}  |  Sharpe SAA = {sr_S:.2f}  ->  SR_P = {SR_P:.2f}")

# Eq. 16 holds only at the optimal size w_A*. At the size we actually hold (w_A from §8, capped by the TE budget):
mu_N = mu_S + w_A * (alpha_A + (beta_A - 1) * mu_S)     # E(R_new^e) = (1 - w_A) E(R_S^e) + w_A E(R_A^e)
print(f"Expected Sharpe at the chosen w_A = {w_A:.1%}: {mu_N / sigma_N:.2f}")


,alpha,sigma2(eps),weight in A,IR_i
NKE,-0.2302,0.0734,0.0000,-0.8497
HD,-0.5770,0.0468,0.0000,-2.6681
GIS,0.0443,0.0443,1.0000,0.2105


IR_A = 0.21  |  Sharpe SAA = 0.27  ->  SR_P = 0.34
Expected Sharpe at the chosen w_A = 18.6%: 0.33


## 10. Portfolio description and trades (for the report and PRTU) §10

**Final weights.** The SAA has $J$ ETFs with weights $w^{SAA}_j$ (summing to 1), and the active portfolio A has $N$ stocks with Treynor-Black weights $w_i$ (summing to 1). We keep $(1-w_A)$ in the SAA and put $w_A$ in A:

$$w^{new}_j = (1-w_A)\,w^{SAA}_j \quad (j = 1,\dots,J), \qquad w^{new}_i = w_A\,w_i \quad (i = 1,\dots,N)$$

$$\sum_j w^{new}_j + \sum_i w^{new}_i = (1-w_A) + w_A = 1$$

Every ETF is cut by the same factor $(1-w_A)$, so the **SAA mix does not change** (only its size). The $N$ stocks share $w_A$ in the Treynor-Black proportions.


**Number of shares:**

$$\text{shares} = \left\lfloor \frac{w^{new} \cdot \text{NAV}}{P} \right\rfloor$$

Shares are rounded down; what is left stays in **cash**. In the challenge use the **NAV of your portfolio** and the **closing prices of Fri 6 Nov** (not 100M and today's prices), then enter the positions in PRTU.


In [12]:
w_new = pd.concat([(1 - w_A) * W_SAA, w_A * w])    # final weights: SAA scaled down + stocks
px    = close[ETFS + STOCKS].ffill().iloc[-1]       # in the challenge: closing prices of Friday 6 November

port = pd.DataFrame({"weight": w_new, "price": px})
port["shares"] = np.floor(port["weight"] * NAV / port["price"])          # NAV of 6 Nov, not 100M
port["USD"]    = port["shares"] * port["price"]
port.loc["CASH", ["weight", "USD"]] = [1 - port["weight"].sum(), NAV - port["USD"].sum()]

print("Price date:", close.index[-1].date(), f"| NAV = USD {NAV:,.0f} | Ex-ante TE vs SAA: {te:.2%}")
port

Price date: 2026-10-28 | NAV = USD 100,000,000 | Ex-ante TE vs SAA: 4.00%


,weight,price,shares,USD
SPY,0.0700,770.8400,9077.0000,6996914.9238
EFA,0.0602,103.2700,58302.0000,6020847.3443
EEM,0.0570,63.1350,90211.0000,5695471.3336
IEF,0.1440,90.9000,158431.0000,14401378.1417
LQD,0.1009,104.2350,96792.0000,10089114.1791
TIP,0.1627,106.1499,153300.0000,16272780.0293
GLD,0.0789,370.3600,21309.0000,7892000.9279
DBC,0.1294,33.9050,381563.0000,12936893.0492
NKE,0.0000,37.5450,0.0000,0.0000
HD,0.0000,320.3950,0.0000,0.0000



**Reading the results:** $w_A = 18.5\%$ with A = 100% GIS → each SAA ETF is multiplied by 0.815 (e.g. TIP 20% → 16.3%), GIS gets 18.5% (550,369 shares), NKE and HD 0, cash 1.1%. Ex-ante TE vs the SAA: 4.0%.